# 第 6 课 · HTML 结构与树（纯观察，不引入解析库）

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 这一课**几乎不引入任何新工具**，只做一件事：把本地的 `books_index.html` 当成一篇文章逐字读熟，
> 并亲手把它画成一棵多叉树。你在 C++ 竞赛里写过无数次树的遍历，这一课就是把网页翻译成那棵树。

## 6.1 先把网页当文本读：标签、属性、文本是什么

**生活化类比（快递面单）**：一张面单上有「收件人栏」「地址栏」「电话栏」。
- 「栏」本身就是**标签**，告诉你这一块是干什么的；
- 栏旁边贴的小字备注（如「易碎」）就是**属性**；
- 栏里真正写的字（收件人姓名）就是**文本**。

**是什么**：HTML 由三样东西组成——
- **标签（tag）**：`<h1>`、`<p>`、`<a>`，成对出现，告诉浏览器「这一块是什么」；
- **属性（attribute）**：写在开始标签里，如 `<a href="..." title="书名">`；
- **文本（text）**：两个标签中间夹的真正内容。

**为什么需要**：爬虫要抠的「数据」，要么在文本里（价格、书名），要么在属性里（链接、title）。先认清这三样，后面才知道去哪抠。

In [1]:
# 第 6 课只用到 Python 内置库，不需要安装任何第三方包
from pathlib import Path          # 拼本地文件路径（每本 notebook 都要重新 import）
from html.parser import HTMLParser  # Python 内置的“事件式”解析器，6.4 节演示用

DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
html_text = (DATA_DIR / 'books_index.html').read_text(encoding='utf-8')
print('类型：', type(html_text))   # <class 'str'>，网页就是一段字符串
print(html_text)                 # 把整页原始文本打印出来，用眼睛从上往下读一遍

类型： <class 'str'>
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <title>在线书店 · 全部书籍（第 1 页）</title>
    <style>
        body { font-family: "Microsoft YaHei", sans-serif; margin: 2rem; background: #fafafa; }
        .product_pod { border: 1px solid #ddd; padding: 1rem; margin: 0.8rem 0; background: #fff; border-radius: 6px; }
        .product_pod h3 a { color: #0066c0; text-decoration: none; }
        .price_color { color: #b10000; font-weight: bold; }
        .instock { color: #2a9d2a; }
        .star-rating.One { color: #e6a23c; }
        .pagination { margin-top: 1.5rem; }
        .pagination .current { font-weight: bold; margin: 0 0.3rem; }
        .pagination a { margin: 0 0.3rem; }
    </style>
</head>
<body>
    <header>
        <h1>在线书店（本地夹具，模拟 books.toscrape.com）</h1>
        <p class="breadcrumb">首页 / 书籍 / 第 1 页</p>
    </header>

    <section class="product_list">
        <article class="product_pod">
            <h3><a href="catalogue/a-light-in-th

In [2]:
# 用最朴素的字符串切片，亲眼看看“一个标签长什么样”
start = html_text.find('<a href=')        # 找到第一个 <a href= 出现的位置
segment = html_text[start:start+120]       # 从那里往后切 120 个字符
print(segment)
# 对着看：<a ...> 是开始标签，title="..." 是属性，>后面、</a>之前 才是文本

<a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the Attic</a></h3>
    


## 6.2 嵌套：标签一层套一层，于是有了「父子 / 兄弟」

**生活化类比**：电脑的文件夹——`D盘` 里放 `复习资料`，`复习资料` 里放 `竞赛`，`竞赛` 里放 `代码.cpp`。
文件夹套文件夹，就是天然的树。HTML 一模一样：`<html>` 里套 `<body>`，`<body>` 里套 `<section>`，`<section>` 里套 `<article>`。

**是什么**：
- 直接套在你里面的标签，是你的**子节点**；你是它们的**父节点**；
- 同一个父亲下的多个节点，互为**兄弟节点**（4 本书就是 4 个兄弟）。

**为什么需要**：网页里同名标签到处都是（几十个 `<p>`），光靠名字分不清。
「它是谁的儿子、有哪些兄弟」才是定位数据的关键。

In [3]:
# 数一下“开始标签”和“结束标签”是否成对——嵌套的前提就是成对出现
print('article 开始标签个数：', html_text.count('<article'))    # 4
print('article 结束标签个数：', html_text.count('</article>'))  # 4
# 4 个开始、4 个结束 = 4 本书各自是一个完整、独立、互不重叠的盒子

article 开始标签个数： 4
article 结束标签个数： 4


## 6.3 本课最关键的动手：把 books_index 亲手画成一棵多叉树

打开 `books_index.html`，对照下面这棵树读。**html 是根**，它有两个儿子 `head` 和 `body`；
`body` 有三个儿子 `header`、`section.product_list`、`nav.pagination`；
`section` 下面并排挂着 **4 个 `article.product_pod`（兄弟节点）**，每个里面再挂 `h3`、`p.star-rating`、`p.price_color`、`p.instock`。

```
<html>
├─ <head>
│   ├─ <meta charset>
│   ├─ <title> 文本“在线书店 · 全部书籍（第 1 页）”
│   └─ <style> …（CSS，爬虫一般不碰）
└─ <body>
    ├─ <header>
    │   ├─ <h1> 文本“在线书店（本地夹具…）”
    │   └─ <p class="breadcrumb"> 文本“首页 / 书籍 / 第 1 页”
    ├─ <section class="product_list">
    │   ├─ <article class="product_pod">   ← 书1：A Light in the Attic / ¥51.77 / Three
    │   │   ├─ <h3>
    │   │   │   └─ <a href="…" title="A Light in the Attic"> 文本“A Light in the Attic”
    │   │   ├─ <p class="star-rating Three"> 文本“星级：Three”
    │   │   ├─ <p class="price_color">       文本“¥51.77”
    │   │   └─ <p class="instock availability"> 文本“有库存（22 本）”
    │   ├─ <article class="product_pod">   ← 书2（结构完全一样）
    │   ├─ <article class="product_pod">   ← 书3
    │   └─ <article class="product_pod">   ← 书4
    └─ <nav class="pagination">
        ├─ <span class="current"> 文本“第 1 页 / 共 2 页”
        └─ <a href="books_page2.html" rel="next"> 文本“下一页 →”
```

> **一句话打通任督二脉**：在网页上「找到我要的那个节点」，就是你图论里的**树遍历 / 节点查找**，
> 只不过这棵树的节点类型换成了 `<article>`、`<p>` 而已。后面三节课学的所有工具，都是在这棵树上帮你找节点。

In [4]:
# 把上面这棵树用程序再“打印”一遍，加深印象（内容是硬编码的示意图）
tree = """<html>
├─ <head> …（meta / title / style）
└─ <body>
    ├─ <header> …（h1、p.breadcrumb）
    ├─ <section class="product_list">
    │   ├─ <article class="product_pod">   # 书1
    │   ├─ <article class="product_pod">   # 书2
    │   ├─ <article class="product_pod">   # 书3
    │   └─ <article class="product_pod">   # 书4
    └─ <nav class="pagination"> …（span.current、a[rel=next]）"""
print(tree)
print('爬虫要做的，就是在这棵多叉树上定位节点——和你竞赛里遍历树没有本质区别。')

<html>
├─ <head> …（meta / title / style）
└─ <body>
    ├─ <header> …（h1、p.breadcrumb）
    ├─ <section class="product_list">
    │   ├─ <article class="product_pod">   # 书1
    │   ├─ <article class="product_pod">   # 书2
    │   ├─ <article class="product_pod">   # 书3
    │   └─ <article class="product_pod">   # 书4
    └─ <nav class="pagination"> …（span.current、a[rel=next]）
爬虫要做的，就是在这棵多叉树上定位节点——和你竞赛里遍历树没有本质区别。


## 6.4 「解析」到底在干嘛：用内置 html.parser 看事件流

**生活化类比**：想象有人拿着稿子从上往下大声念——「开始标签 h3」「开始标签 a」「文本 A Light…」「结束标签 a」……
他不整理书架，只是**遇到什么就喊什么**。这种「遇到即喊」的东西叫**事件**。

**是什么**：Python 内置的 `HTMLParser` 会从上到下扫字符串，每遇到开始标签、文本、结束标签，就自动调用对应的方法。
你只要 override（重写）这几个方法，就能看到完整事件流。

**为什么需要**：第 7 课的 BeautifulSoup 也是在做同一件事，但它悄悄把事件整理成了你能随便查的树。
先看一眼「原始事件」，你就不会觉得 BeautifulSoup 是黑魔法了。

In [5]:
class EventPrinter(HTMLParser):
    def handle_starttag(self, tag, attrs):            # 每遇到一个开始标签，自动调用
        print('▶ 开始标签:', tag, '| 属性:', dict(attrs))  # attrs 是 [(名,值)…]，转字典好看

    def handle_data(self, data):                       # 遇到标签之间的文本时调用
        text = data.strip()
        if text:                                       # 换行、缩进这类空白就跳过
            print('    文本:', repr(text))

    def handle_endtag(self, tag):                       # 遇到结束标签时调用
        print('◀ 结束标签:', tag)

parser = EventPrinter()                 # 新建一个“播报器”
parser.feed(html_text[:500])            # 只喂前 500 个字符，免得刷屏；它会依次播报事件

▶ 开始标签: html | 属性: {'lang': 'zh-CN'}
▶ 开始标签: head | 属性: {}
▶ 开始标签: meta | 属性: {'charset': 'UTF-8'}
▶ 开始标签: title | 属性: {}
    文本: '在线书店 · 全部书籍（第 1 页）'
◀ 结束标签: title
▶ 开始标签: style | 属性: {}


## 6.5 随堂小练习（自己动手，空格里写）

In [6]:
# TODO：用 .count() 数一下 books_index.html 里一共有多少个 <p 开始标签

In [7]:
# TODO：在纸上把 quotes.html 的主体画成一棵 ASCII 树（提示：3 个 div.quote 是兄弟）

## 来源与延伸

- MDN《HTML 入门》：https://developer.mozilla.org/zh-CN/docs/Learn/HTML/Introduction_to_HTML/Getting_started
- Python 内置 html.parser 文档：https://docs.python.org/zh-cn/3/library/html.parser.html
- 下一课：BeautifulSoup 把这一路事件整理成一棵可以随手查找的树。